In [2]:
import pandas as pd

# Load the raw ball-by-ball data
df = pd.read_csv('train_IPL.csv')

# Print a success message and show the size of the dataset
print("Success! Data loaded.")
print("Total rows and columns:", df.shape)

# Look at the first 5 rows to understand the structure
display(df.head())

C:\Users\Narasimha\AppData\Local\Temp\ipykernel_21792\2308635844.py:4: DtypeWarning: Columns (0: result_type, 1: season) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('train_IPL.csv')


Success! Data loaded.
Total rows and columns: (272704, 38)


,Match ID,Date,Venue,Bat First,Bat Second,Innings,Over,Ball,Batter,Non Striker,...,Player Out Runs,Player Out Balls Faced,Bowler Runs Conceded,Valid Ball,toss_winner,toss_decision,city,result_type,season,match_won_by
0,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,1,SC Ganguly,BB McCullum,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
1,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,2,BB McCullum,SC Ganguly,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
2,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,3,BB McCullum,SC Ganguly,...,NaN,NaN,1,0,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
3,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,3,BB McCullum,SC Ganguly,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
4,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,4,BB McCullum,SC Ganguly,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders


In [4]:
print(df.columns.tolist())

['Match ID', 'Date', 'Venue', 'Bat First', 'Bat Second', 'Innings', 'Over', 'Ball', 'Batter', 'Non Striker', 'Bowler', 'Batter Runs', 'Extra Runs', 'Runs From Ball', 'Ball Rebowled', 'Extra Type', 'Wicket', 'Dismissal Method', 'Player Out', 'Innings Runs', 'Innings Wickets', 'Target Score', 'Runs to Get', 'Balls Remaining', 'Total Batter Runs', 'Total Non Striker Runs', 'Batter Balls Faced', 'Non Striker Balls Faced', 'Player Out Runs', 'Player Out Balls Faced', 'Bowler Runs Conceded', 'Valid Ball', 'toss_winner', 'toss_decision', 'city', 'result_type', 'season', 'match_won_by']


In [6]:
print("Aggregating ball-by-ball data...")

# 1. Exact match identifier from your print output
match_column_name = 'Match ID' 

# Get the final winner for each match
match_winners = df[[match_column_name, 'match_won_by']].drop_duplicates()

# 2. Filter for only the first innings (notice the capital 'I' and plural 's')
first_innings = df[df['Innings'] == 1]

# Calculate the total runs scored in the first innings
# Using the exact team and runs column names from your list
first_innings_runs = first_innings.groupby(
    [match_column_name, 'Bat First', 'Bat Second']
)['Runs From Ball'].sum().reset_index()

# Rename the runs column so it makes sense for our model
first_innings_runs.rename(columns={'Runs From Ball': 'first_innings_score'}, inplace=True)

# 3. Merge them together into our final training dataset
match_data = pd.merge(first_innings_runs, match_winners, on=match_column_name, how='left')

# 4. Drop matches with no clear winner (e.g., washed out by rain)
match_data = match_data.dropna(subset=['match_won_by'])

print("Successfully created Match-Level data!")
print("New Dataset Shape:", match_data.shape)
display(match_data.head())

Aggregating ball-by-ball data...
Successfully created Match-Level data!
New Dataset Shape: (1142, 5)


,Match ID,Bat First,Bat Second,first_innings_score,match_won_by
0,335982,Kolkata Knight Riders,Royal Challengers Bangalore,222,Kolkata Knight Riders
1,335983,Chennai Super Kings,Kings XI Punjab,240,Chennai Super Kings
2,335984,Rajasthan Royals,Delhi Daredevils,129,Delhi Daredevils
3,335985,Mumbai Indians,Royal Challengers Bangalore,165,Royal Challengers Bangalore
4,335986,Deccan Chargers,Kolkata Knight Riders,110,Kolkata Knight Riders


In [8]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

print("Preparing data for Machine Learning...")

# --- THE FIX: Remove matches where the winner is 'Unknown' ---
match_data = match_data[match_data['match_won_by'] != 'Unknown']
# -------------------------------------------------------------

# 1. Create a translator (LabelEncoder) for team names
team_encoder = LabelEncoder()

# Gather all unique team names from both columns to ensure the ID is consistent 
all_teams = pd.concat([match_data['Bat First'], match_data['Bat Second']]).unique()
team_encoder.fit(all_teams)

# 2. Translate the team text into numbers
match_data['Bat_First_ID'] = team_encoder.transform(match_data['Bat First'])
match_data['Bat_Second_ID'] = team_encoder.transform(match_data['Bat Second'])
match_data['Winner_ID'] = team_encoder.transform(match_data['match_won_by'])

# 3. Define our Features (X) and Target (y)
# X contains the clues: Who is batting first, second, and how many runs did they score?
X = match_data[['Bat_First_ID', 'Bat_Second_ID', 'first_innings_score']]

# y contains the answer key: Who actually won?
y = match_data['Winner_ID']

print("Data successfully translated to numbers!")
print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

# Let's peek at the numerical version of our data
display(X.head())

Preparing data for Machine Learning...
Data successfully translated to numbers!
Features (X) shape: (1124, 3)
Target (y) shape: (1124,)


,Bat_First_ID,Bat_Second_ID,first_innings_score
0,8,16,222
1,0,6,240
2,13,3,129
3,10,16,165
4,1,8,110


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss

print("Step 4: Training the Machine Learning Model...")

# 1. Split data into Training (80%) and Validation (20%) sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Initialize our algorithm
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# 3. Train the model using our historical features and answers
print("Learning from historical IPL matches...")
model.fit(X_train, y_train)

# 4. Make predictions on our validation set using PROBABILITIES
val_predictions = model.predict_proba(X_val)

# 5. Calculate our Log Loss Score
# THE FIX: We explicitly pass the model's known classes so log_loss doesn't get confused
score = log_loss(y_val, val_predictions, labels=model.classes_)

print("\n--- Training Complete! ---")
print(f"Validation Log Loss: {score:.4f}")

# Let's see what the actual probability outputs look like for the first 3 matches
print("\nSample Probabilities (First 3 validation matches):")
print(val_predictions[:3])

Step 4: Training the Machine Learning Model...
Learning from historical IPL matches...

--- Training Complete! ---
Validation Log Loss: 1.7972

Sample Probabilities (First 3 validation matches):
[[2.27884672e-02 2.85161909e-02 4.20663717e-02 5.40728523e-02
  1.43294833e-02 2.65644714e-02 6.96193517e-02 5.28916988e-03
  1.49199303e-01 8.36294872e-02 1.22415634e-01 1.08867735e-02
  6.63061302e-02 1.41206249e-01 1.26615408e-02 9.17039037e-03
  9.85339400e-02 1.02569598e-02 3.24872335e-02]
 [1.73401091e-02 1.52641355e-02 1.60175931e-02 2.50050808e-02
  5.84097025e-03 9.03404151e-03 3.94790743e-02 2.98353755e-03
  7.86789223e-02 1.01967146e-02 9.47884707e-02 8.53525863e-03
  3.62445433e-02 1.25286822e-01 3.19492473e-02 1.04371071e-02
  3.13198303e-01 1.92656251e-02 1.40454444e-01]
 [1.24488628e-01 3.93201072e-02 2.84235247e-02 4.02249839e-02
  2.99106378e-03 3.20907215e-02 5.41945019e-02 4.68882785e-02
  1.66397130e-01 7.03629729e-03 2.54540267e-01 1.66490411e-02
  3.72998565e-02 4.55326395

In [12]:
import pandas as pd

print("Step 5: Loading Competition Test Data...")

# Reading the two specific files you just uploaded
test_matches = pd.read_csv('public_lb_matches.csv')
submission_template = pd.read_csv('sample_submission.csv')

print("Successfully loaded competition files!")
print("\n--- Exact Column Names in Test Data ---")
print(test_matches.columns.tolist())

# Display the first few rows to see what the upcoming matches look like
display(test_matches.head())

Step 5: Loading Competition Test Data...
Successfully loaded competition files!

--- Exact Column Names in Test Data ---
['match_id', 'date', 'season', 'team_a', 'team_b', 'venue', 'city', 'toss_winner', 'toss_decision']


,match_id,date,season,team_a,team_b,venue,city,toss_winner,toss_decision
0,1473488,2025-05-02,2025,Gujarat Titans,Sunrisers Hyderabad,Narendra Modi Stadium,Ahmedabad,Sunrisers Hyderabad,field
1,1473489,2025-05-03,2025,Royal Challengers Bengaluru,Chennai Super Kings,M Chinnaswamy Stadium,Bengaluru,Chennai Super Kings,field
2,1473490,2025-05-04,2025,Kolkata Knight Riders,Rajasthan Royals,Eden Gardens,Kolkata,Kolkata Knight Riders,bat
3,1473491,2025-05-04,2025,Punjab Kings,Lucknow Super Giants,Himachal Pradesh Cricket Association Stadium,Dharamsala,Lucknow Super Giants,field
4,1473492,2025-05-05,2025,Delhi Capitals,NaN,Rajiv Gandhi International Stadium,Hyderabad,Sunrisers Hyderabad,field


In [16]:
print(test_matches.columns.tolist())

['match_id', 'date', 'season', 'team_a', 'team_b', 'venue', 'city', 'toss_winner', 'toss_decision']


In [18]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

print("Step 6: Generating Final Kaggle Predictions...")

team_1_col = 'team_a' 
team_2_col = 'team_b'

# --- THE FIX: Handle missing (TBD) teams in playoff matches ---
# We grab the first valid team name our encoder knows and use it to fill the blanks
placeholder_team = team_encoder.classes_[0]
test_matches[team_1_col] = test_matches[team_1_col].fillna(placeholder_team)
test_matches[team_2_col] = test_matches[team_2_col].fillna(placeholder_team)
# --------------------------------------------------------------

# 1. Retrain the model using ONLY pre-match information
X_final_train = match_data[['Bat_First_ID', 'Bat_Second_ID']]
y_final_train = match_data['Winner_ID']

final_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
final_model.fit(X_final_train, y_final_train)
print("Final model trained on pre-match data!")

# 2. Prepare the upcoming Kaggle matches
test_matches['Bat_First_ID'] = team_encoder.transform(test_matches[team_1_col])
test_matches['Bat_Second_ID'] = team_encoder.transform(test_matches[team_2_col])
X_test = test_matches[['Bat_First_ID', 'Bat_Second_ID']]

# 3. Generate Probabilities
test_predictions = final_model.predict_proba(X_test)

# 4. Format into Kaggle's Exact Structure
team_names = team_encoder.inverse_transform(final_model.classes_)
submission_df = pd.DataFrame(test_predictions, columns=team_names)

# Add the Match ID back to the very front using the exact test column name
submission_df.insert(0, 'match_id', test_matches['match_id'])

# 5. Save to a CSV file!
submission_df.to_csv('my_first_submission.csv', index=False)

print("\nSUCCESS! 'my_first_submission.csv' has been created.")
display(submission_df.head())

Step 6: Generating Final Kaggle Predictions...
Final model trained on pre-match data!

SUCCESS! 'my_first_submission.csv' has been created.


,match_id,Chennai Super Kings,Deccan Chargers,Delhi Capitals,Delhi Daredevils,Gujarat Lions,Gujarat Titans,Kings XI Punjab,Kochi Tuskers Kerala,Kolkata Knight Riders,Lucknow Super Giants,Mumbai Indians,Pune Warriors,Punjab Kings,Rajasthan Royals,Rising Pune Supergiant,Rising Pune Supergiants,Royal Challengers Bangalore,Royal Challengers Bengaluru,Sunrisers Hyderabad
0,1473488,0.005110,0.021972,0.047132,0.050701,0.002036,0.021931,0.078203,0.001471,0.126137,0.017944,0.054354,0.002518,0.013767,0.052700,0.009964,0.007645,0.132751,0.034200,0.319465
1,1473489,0.488777,0.000532,0.002446,0.002278,0.001029,0.003456,0.004198,0.000040,0.004646,0.000485,0.003450,0.000535,0.004731,0.019714,0.000079,0.000000,0.114835,0.315404,0.033365
2,1473490,0.001963,0.026317,0.049684,0.068793,0.008533,0.024567,0.073268,0.002695,0.147819,0.034195,0.111885,0.010211,0.043921,0.256121,0.014231,0.005382,0.075968,0.012402,0.032043
3,1473491,0.005655,0.018991,0.040656,0.053936,0.014244,0.040127,0.067169,0.007447,0.162772,0.048450,0.196683,0.021000,0.041658,0.129889,0.012527,0.001496,0.088073,0.011654,0.037570
4,1473492,0.486556,0.041340,0.330257,0.049247,0.000469,0.021495,0.028771,0.000452,0.012066,0.002108,0.015383,0.001561,0.001062,0.004023,0.000143,0.000370,0.003034,0.000563,0.001100


In [21]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

print("Fixing data types and row count mismatch for Kaggle...")

# 1. Retrain the baseline model using match data
X_final_train = match_data[['Bat_First_ID', 'Bat_Second_ID']]
y_final_train = match_data['Winner_ID']

final_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
final_model.fit(X_final_train, y_final_train)

# 2. Map all 19 team names exactly as the model expects them
team_names = team_encoder.inverse_transform(final_model.classes_)

# 3. Create a brand-new submission structure using the official template
submission_df = pd.DataFrame(columns=team_names)
submission_df.insert(0, 'match_id', submission_template['match_id'])

# 4. Generate the probabilities for the test dataset
test_predictions = final_model.predict_proba(X_test)
predicted_df = pd.DataFrame(test_predictions, columns=team_names)
predicted_df['match_id'] = test_matches['match_id']

# --- THE FINAL FIX: Force both match_id columns to be TEXT (strings) ---
submission_df['match_id'] = submission_df['match_id'].astype(str)
predicted_df['match_id'] = predicted_df['match_id'].astype(str)
# -----------------------------------------------------------------------

# 5. Merge our predictions onto the mandatory 53-row template
submission_df = pd.merge(submission_df[['match_id']], predicted_df, on='match_id', how='left')

# 6. Fill any blank matches with an even probability split
submission_df = submission_df.fillna(1.0 / len(team_names))

# 7. Save the final file over your old one
submission_df.to_csv('my_first_submission.csv', index=False)

print("\nSUCCESS! New 53-row file generated safely.")
print("Total rows in file:", len(submission_df))
display(submission_df.head())

Fixing data types and row count mismatch for Kaggle...

SUCCESS! New 53-row file generated safely.
Total rows in file: 53


,match_id,Chennai Super Kings,Deccan Chargers,Delhi Capitals,Delhi Daredevils,Gujarat Lions,Gujarat Titans,Kings XI Punjab,Kochi Tuskers Kerala,Kolkata Knight Riders,Lucknow Super Giants,Mumbai Indians,Pune Warriors,Punjab Kings,Rajasthan Royals,Rising Pune Supergiant,Rising Pune Supergiants,Royal Challengers Bangalore,Royal Challengers Bengaluru,Sunrisers Hyderabad
0,1473488,0.005110,0.021972,0.047132,0.050701,0.002036,0.021931,0.078203,0.001471,0.126137,0.017944,0.054354,0.002518,0.013767,0.052700,0.009964,0.007645,0.132751,0.034200,0.319465
1,1473489,0.488777,0.000532,0.002446,0.002278,0.001029,0.003456,0.004198,0.000040,0.004646,0.000485,0.003450,0.000535,0.004731,0.019714,0.000079,0.000000,0.114835,0.315404,0.033365
2,1473490,0.001963,0.026317,0.049684,0.068793,0.008533,0.024567,0.073268,0.002695,0.147819,0.034195,0.111885,0.010211,0.043921,0.256121,0.014231,0.005382,0.075968,0.012402,0.032043
3,1473491,0.005655,0.018991,0.040656,0.053936,0.014244,0.040127,0.067169,0.007447,0.162772,0.048450,0.196683,0.021000,0.041658,0.129889,0.012527,0.001496,0.088073,0.011654,0.037570
4,1473492,0.486556,0.041340,0.330257,0.049247,0.000469,0.021495,0.028771,0.000452,0.012066,0.002108,0.015383,0.001561,0.001062,0.004023,0.000143,0.000370,0.003034,0.000563,0.001100


In [22]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

print("Building bulletproof submission file...")

# 1. Retrain the baseline model 
X_final_train = match_data[['Bat_First_ID', 'Bat_Second_ID']]
y_final_train = match_data['Winner_ID']

final_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
final_model.fit(X_final_train, y_final_train)

# 2. Get predictions and map them to team names
team_names = team_encoder.inverse_transform(final_model.classes_)
test_predictions = final_model.predict_proba(X_test)

predicted_df = pd.DataFrame(test_predictions, columns=team_names)
predicted_df['match_id'] = test_matches['match_id'].astype(str)

# --- THE BULLETPROOF FIX ---
# 3. Use Kaggle's EXACT template to guarantee column order
final_submission = submission_template.copy()
final_submission['match_id'] = final_submission['match_id'].astype(str)

# 4. Surgically update the template with our predictions
final_submission.set_index('match_id', inplace=True)
predicted_df.set_index('match_id', inplace=True)

final_submission.update(predicted_df)

# Reset the index so match_id becomes a normal column again
final_submission.reset_index(inplace=True)
# ---------------------------

# 5. Handle any completely blank playoff rows to prevent Kaggle from crashing
# We assign an equal probability (1 divided by 19 teams) to avoid dividing by zero
default_prob = 1.0 / len(team_names)
final_submission = final_submission.fillna(default_prob)

# 6. Save the final file!
final_submission.to_csv('my_first_submission.csv', index=False)

print("\nSUCCESS! Bulletproof file generated.")
display(final_submission.head())

Building bulletproof submission file...

SUCCESS! Bulletproof file generated.


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.25,0.25,0.25,0.25
1,1473489,0.25,0.25,0.25,0.25
2,1473490,0.25,0.25,0.25,0.25
3,1473491,0.25,0.25,0.25,0.25
4,1473492,0.25,0.25,0.25,0.25


In [23]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

print("Phase 2: Adding 'Toss' Features to Improve Score...")

# 1. Bring Toss Data into our historical match dataset
toss_data = df[['Match ID', 'toss_winner', 'toss_decision']].drop_duplicates()
updated_match_data = pd.merge(match_data, toss_data, on='Match ID', how='left')

# Drop any matches missing toss data just to be safe
updated_match_data = updated_match_data.dropna(subset=['toss_winner', 'toss_decision'])

# 2. Translate the Toss Winner (using our existing team_encoder)
updated_match_data['Toss_Winner_ID'] = team_encoder.transform(updated_match_data['toss_winner'])

# 3. Create a NEW translator for Toss Decision ('bat' or 'field' becomes 0 or 1)
decision_encoder = LabelEncoder()
updated_match_data['Toss_Decision_ID'] = decision_encoder.fit_transform(updated_match_data['toss_decision'])

# 4. Define our NEW Upgraded Features (X)
X_train_upgraded = updated_match_data[['Bat_First_ID', 'Bat_Second_ID', 'Toss_Winner_ID', 'Toss_Decision_ID']]
y_train_upgraded = updated_match_data['Winner_ID']

# 5. Train the upgraded V2 model
upgraded_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
upgraded_model.fit(X_train_upgraded, y_train_upgraded)
print("Upgraded V2 model trained successfully!")

# 6. Prepare the Kaggle Test Data
# Fill any missing playoff toss data with safe placeholders
placeholder_team = team_encoder.classes_[0]
test_matches['toss_winner'] = test_matches['toss_winner'].fillna(placeholder_team)
test_matches['toss_decision'] = test_matches['toss_decision'].fillna('field') # Default guess

# Translate text to numbers for the test set
test_matches['Toss_Winner_ID'] = team_encoder.transform(test_matches['toss_winner'])
test_matches['Toss_Decision_ID'] = decision_encoder.transform(test_matches['toss_decision'])

X_test_upgraded = test_matches[['Bat_First_ID', 'Bat_Second_ID', 'Toss_Winner_ID', 'Toss_Decision_ID']]

# 7. Generate New Probabilities
test_predictions_upgraded = upgraded_model.predict_proba(X_test_upgraded)
predicted_df_upgraded = pd.DataFrame(test_predictions_upgraded, columns=team_names)
predicted_df_upgraded['match_id'] = test_matches['match_id'].astype(str)

# 8. Surgically update the Kaggle template (The Bulletproof Method)
final_submission_2 = submission_template.copy()
final_submission_2['match_id'] = final_submission_2['match_id'].astype(str)
final_submission_2.set_index('match_id', inplace=True)
predicted_df_upgraded.set_index('match_id', inplace=True)

final_submission_2.update(predicted_df_upgraded)
final_submission_2.reset_index(inplace=True)

# Fill blanks for playoffs
default_prob = 1.0 / len(team_names)
final_submission_2 = final_submission_2.fillna(default_prob)

# 9. Save the NEW submission file
final_submission_2.to_csv('my_second_submission.csv', index=False)

print("\nSUCCESS! 'my_second_submission.csv' generated.")
display(final_submission_2.head())

Phase 2: Adding 'Toss' Features to Improve Score...
Upgraded V2 model trained successfully!

SUCCESS! 'my_second_submission.csv' generated.


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.25,0.25,0.25,0.25
1,1473489,0.25,0.25,0.25,0.25
2,1473490,0.25,0.25,0.25,0.25
3,1473491,0.25,0.25,0.25,0.25
4,1473492,0.25,0.25,0.25,0.25


In [25]:
print(df.columns.tolist())

['Match ID', 'Date', 'Venue', 'Bat First', 'Bat Second', 'Innings', 'Over', 'Ball', 'Batter', 'Non Striker', 'Bowler', 'Batter Runs', 'Extra Runs', 'Runs From Ball', 'Ball Rebowled', 'Extra Type', 'Wicket', 'Dismissal Method', 'Player Out', 'Innings Runs', 'Innings Wickets', 'Target Score', 'Runs to Get', 'Balls Remaining', 'Total Batter Runs', 'Total Non Striker Runs', 'Batter Balls Faced', 'Non Striker Balls Faced', 'Player Out Runs', 'Player Out Balls Faced', 'Bowler Runs Conceded', 'Valid Ball', 'toss_winner', 'toss_decision', 'city', 'result_type', 'season', 'match_won_by']


In [26]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

print("Phase 3: Adding 'Venue' Features to Improve Score...")

# 1. Bring Venue Data into our historical dataset
# Pull using the exact spelling 'Venue' with a capital V
venue_data = df[['Match ID', 'Venue']].drop_duplicates()

# Rename it to lowercase 'venue' so it matches Kaggle's test file exactly!
venue_data = venue_data.rename(columns={'Venue': 'venue'})

match_data_v3 = pd.merge(updated_match_data, venue_data, on='Match ID', how='left')

# Drop any matches missing venue data just to keep things clean
match_data_v3 = match_data_v3.dropna(subset=['venue'])

# 2. Translate the Stadium Names into numbers
venue_encoder = LabelEncoder()
match_data_v3['Venue_ID'] = venue_encoder.fit_transform(match_data_v3['venue'])

# 3. Define our NEW Upgraded Features (Now we have 5 clues!)
X_train_v3 = match_data_v3[['Bat_First_ID', 'Bat_Second_ID', 'Toss_Winner_ID', 'Toss_Decision_ID', 'Venue_ID']]
y_train_v3 = match_data_v3['Winner_ID']

# 4. Train the V3 model
model_v3 = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_v3.fit(X_train_v3, y_train_v3)
print("V3 model trained successfully with Venue data!")

# 5. Prepare the Kaggle Test Data
# Find the most common stadium to use as a safe placeholder for missing data
most_common_stadium = match_data_v3['venue'].mode()[0]
test_matches['venue'] = test_matches['venue'].fillna(most_common_stadium)

# Handle any brand-new stadiums in the test set by safely replacing them with the most common one
test_matches['venue'] = test_matches['venue'].apply(lambda x: x if x in venue_encoder.classes_ else most_common_stadium)
test_matches['Venue_ID'] = venue_encoder.transform(test_matches['venue'])

X_test_v3 = test_matches[['Bat_First_ID', 'Bat_Second_ID', 'Toss_Winner_ID', 'Toss_Decision_ID', 'Venue_ID']]

# 6. Generate New Probabilities
test_predictions_v3 = model_v3.predict_proba(X_test_v3)
predicted_df_v3 = pd.DataFrame(test_predictions_v3, columns=team_names)
predicted_df_v3['match_id'] = test_matches['match_id'].astype(str)

# 7. Surgically update the Kaggle template
final_submission_3 = submission_template.copy()
final_submission_3['match_id'] = final_submission_3['match_id'].astype(str)
final_submission_3.set_index('match_id', inplace=True)
predicted_df_v3.set_index('match_id', inplace=True)

final_submission_3.update(predicted_df_v3)
final_submission_3.reset_index(inplace=True)

# Fill blanks for playoffs
final_submission_3 = final_submission_3.fillna(default_prob)

# 8. Save the NEW submission file
final_submission_3.to_csv('my_third_submission.csv', index=False)

print("\nSUCCESS! 'my_third_submission.csv' generated.")
display(final_submission_3.head())

Phase 3: Adding 'Venue' Features to Improve Score...
V3 model trained successfully with Venue data!

SUCCESS! 'my_third_submission.csv' generated.


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.25,0.25,0.25,0.25
1,1473489,0.25,0.25,0.25,0.25
2,1473490,0.25,0.25,0.25,0.25
3,1473491,0.25,0.25,0.25,0.25
4,1473492,0.25,0.25,0.25,0.25


In [27]:
import pandas as pd
# Import our new heavy-hitting algorithm
from xgboost import XGBClassifier

print("Phase 4: The Secret Sauce - Upgrading to XGBoost...")

# 1. Initialize the XGBoost Classifier
# We tune the 'learning_rate' so it doesn't learn too fast and overfit the data
xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)

# 2. Train the model using our best V3 features!
xgb_model.fit(X_train_v3, y_train_v3)
print("XGBoost model trained successfully!")

# 3. Generate New Probabilities using the V3 test data
test_predictions_v4 = xgb_model.predict_proba(X_test_v3)
predicted_df_v4 = pd.DataFrame(test_predictions_v4, columns=team_names)
predicted_df_v4['match_id'] = test_matches['match_id'].astype(str)

# 4. Surgically update the Kaggle template (The Bulletproof Method)
final_submission_4 = submission_template.copy()
final_submission_4['match_id'] = final_submission_4['match_id'].astype(str)
final_submission_4.set_index('match_id', inplace=True)
predicted_df_v4.set_index('match_id', inplace=True)

final_submission_4.update(predicted_df_v4)
final_submission_4.reset_index(inplace=True)

# Fill blanks for the upcoming playoffs
final_submission_4 = final_submission_4.fillna(default_prob)

# 5. Save the NEW submission file
final_submission_4.to_csv('my_fourth_submission_xgb.csv', index=False)

print("\nSUCCESS! 'my_fourth_submission_xgb.csv' generated.")
display(final_submission_4.head())

Phase 4: The Secret Sauce - Upgrading to XGBoost...
XGBoost model trained successfully!

SUCCESS! 'my_fourth_submission_xgb.csv' generated.


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.25,0.25,0.25,0.25
1,1473489,0.25,0.25,0.25,0.25
2,1473490,0.25,0.25,0.25,0.25
3,1473491,0.25,0.25,0.25,0.25
4,1473492,0.25,0.25,0.25,0.25


In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# 1. Load your files (Make sure these match exactly with your directory)
print("Loading data...")
train_df = pd.read_csv('train_IPL.csv')
# Assuming 'public_lb_matches.csv' contains the matches you need to predict
test_df = pd.read_csv('public_lb_matches.csv') 
sample_sub = pd.read_csv('sample_submission.csv')

# 2. Identify Target and Features 
# IMPORTANT: Change 'winner' to whatever your target column is actually named in train_IPL.csv
target_col = 'winner' 

# Map your target to 4 distinct classes (0, 1, 2, 3)
# You will need to adjust the logic based on how ties/no results are labeled in your data
def map_target(row):
    if row['winner'] == row['team1']: return 0
    elif row['winner'] == row['team2']: return 1
    elif row['winner'] == 'Tie': return 2
    else: return 3 # No Result

train_df['target'] = train_df.apply(map_target, axis=1)

# Drop columns that shouldn't be trained on (like IDs or the target itself)
drop_cols = ['match_id', target_col, 'target']
features = [col for col in train_df.columns if col not in drop_cols]

X = train_df[features]
y = train_df['target']
X_test = test_df[features]

# 3. Handle Categorical Data
# XGBoost can handle categories, but Label Encoding is a safe baseline
print("Encoding categorical features...")
label_encoders = {}
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        # Fit on both train and test to avoid unseen label errors
        le.fit(list(X[col].astype(str)) + list(X_test[col].astype(str)))
        X[col] = le.transform(X[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))

# 4. Train the Model for Log Loss
print("Training XGBoost Model...")
model = xgb.XGBClassifier(
    objective='multi:softprob', # Crucial for getting probabilities
    num_class=4,                # Team1, Team2, Tie, No Result
    n_estimators=300,           # Number of trees
    learning_rate=0.03,         # Lower learning rate = more robust
    max_depth=4,                # Shallow trees prevent overfitting
    subsample=0.8,              # Use 80% of data per tree to generalize better
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X, y)

# 5. Generate Raw Probabilities
raw_probs = model.predict_proba(X_test)

# 6. LOG LOSS PROTECTION: Probability Clipping
# Never predict 1.0 or 0.0. This is the secret to a "Strong Submission".
MIN_PROB = 0.02
MAX_PROB = 0.98
clipped_probs = np.clip(raw_probs, MIN_PROB, MAX_PROB)

# 7. RULES COMPLIANCE: Row Normalization
# The rules state rows must sum exactly to 1.0. Clipping messes this up, so we fix it here.
normalized_probs = clipped_probs / clipped_probs.sum(axis=1, keepdims=True)

# 8. Create Final Submission
# Ensure column names exactly match the sample_submission.csv requirements
submission_columns = ['Team1_Win', 'Team2_Win', 'Tie', 'No_Result'] # Adjust if sample_sub has different names
final_sub = pd.DataFrame(normalized_probs, columns=submission_columns)

# Insert the match_id from the sample submission
final_sub.insert(0, 'match_id', sample_sub['match_id'])

# Save the file
file_name = 'my_strong_submission_final.csv'
final_sub.to_csv(file_name, index=False)
print(f"Success! {file_name} is ready to upload.")

Loading data...


KeyError: 'winner'

In [2]:
# Print all column names as a list
print(train_df.columns.tolist())

['Match ID', 'Date', 'Venue', 'Bat First', 'Bat Second', 'Innings', 'Over', 'Ball', 'Batter', 'Non Striker', 'Bowler', 'Batter Runs', 'Extra Runs', 'Runs From Ball', 'Ball Rebowled', 'Extra Type', 'Wicket', 'Dismissal Method', 'Player Out', 'Innings Runs', 'Innings Wickets', 'Target Score', 'Runs to Get', 'Balls Remaining', 'Total Batter Runs', 'Total Non Striker Runs', 'Batter Balls Faced', 'Non Striker Balls Faced', 'Player Out Runs', 'Player Out Balls Faced', 'Bowler Runs Conceded', 'Valid Ball', 'toss_winner', 'toss_decision', 'city', 'result_type', 'season', 'match_won_by']


In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("1. Loading data...")
train_df = pd.read_csv('train_IPL.csv')
test_df = pd.read_csv('public_lb_matches.csv') 
sample_sub = pd.read_csv('sample_submission.csv')

target_col = 'match_won_by'
t1_col = 'Bat First'
t2_col = 'Bat Second'

print("2. Mapping targets...")
def fast_map(row):
    val = str(row[target_col]).strip()
    if val == str(row[t1_col]).strip(): return 0
    elif val == str(row[t2_col]).strip(): return 1
    elif str(row['result_type']).strip().lower() == 'tie': return 2
    else: return 3 

train_df['target'] = train_df.apply(fast_map, axis=1)

print("3. Prepping features...")
drop_cols = ['Match ID', 'match_id', target_col, 'target', 'result_type']
features = [c for c in train_df.columns if c not in drop_cols]

X = train_df[features].copy()
X_test = test_df[features].copy() if features[0] in test_df.columns else test_df.copy()

for col in X.columns:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[X.columns]

print("4. BULLETPROOF ENCODING (Fixing the crash)...")
# Forcefully convert every string/object column to pure integers
for col in X.columns:
    if X[col].dtype == 'object' or 'str' in str(X[col].dtype) or 'string' in str(X[col].dtype):
        le = LabelEncoder()
        # Combine train and test to fit all possible words
        all_words = list(X[col].astype(str)) + list(X_test[col].astype(str))
        le.fit(all_words)
        X[col] = le.transform(X[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))

# Final safety net: Force everything to float
X = X.apply(pd.to_numeric, errors='coerce').fillna(-1)
X_test = X_test.apply(pd.to_numeric, errors='coerce').fillna(-1)

print("5. Training model...")
model = xgb.XGBClassifier(
    objective='multi:softprob', 
    num_class=4, 
    n_estimators=100, 
    max_depth=3, 
    learning_rate=0.05,
    n_jobs=-1
)
model.fit(X, train_df['target'])

print("6. Generating predictions...")
raw_probs = model.predict_proba(X_test)
clipped_probs = np.clip(raw_probs, 0.05, 0.95) 
norm_probs = clipped_probs / clipped_probs.sum(axis=1, keepdims=True)

print("7. Exporting...")
final_sub = pd.DataFrame(norm_probs, columns=['Team1_Win', 'Team2_Win', 'Tie', 'No_Result'])

if len(final_sub) > len(sample_sub):
    final_sub = final_sub.iloc[:len(sample_sub)]
elif len(final_sub) < len(sample_sub):
    padding = pd.DataFrame([[0.45, 0.45, 0.05, 0.05]] * (len(sample_sub) - len(final_sub)), columns=final_sub.columns)
    final_sub = pd.concat([final_sub, padding], ignore_index=True)

final_sub.insert(0, 'match_id', sample_sub['match_id'])
final_sub.to_csv('FINAL_SUBMISSION.csv', index=False)

print("🚀 DONE! UPLOAD 'FINAL_SUBMISSION.csv' IMMEDIATELY!")

1. Loading data...
2. Mapping targets...
3. Prepping features...
4. BULLETPROOF ENCODING (Fixing the crash)...
5. Training model...
6. Generating predictions...
7. Exporting...
🚀 DONE! UPLOAD 'FINAL_SUBMISSION.csv' IMMEDIATELY!


In [1]:
import pandas as pd

# Load the raw match data
train_matches = pd.read_csv('train_IPL.csv')

# Check the shape (Rows, Columns)
print("Train Matches Shape:", train_matches.shape)

# Display the first 3 rows
train_matches.head(3)

C:\Users\Narasimha\AppData\Local\Temp\ipykernel_11616\247671962.py:4: DtypeWarning: Columns (35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  train_matches = pd.read_csv('train_IPL.csv')


Train Matches Shape: (272704, 38)


,Match ID,Date,Venue,Bat First,Bat Second,Innings,Over,Ball,Batter,Non Striker,...,Player Out Runs,Player Out Balls Faced,Bowler Runs Conceded,Valid Ball,toss_winner,toss_decision,city,result_type,season,match_won_by
0,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,1,SC Ganguly,BB McCullum,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
1,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,2,BB McCullum,SC Ganguly,...,NaN,NaN,0,1,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders
2,335982,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,3,BB McCullum,SC Ganguly,...,NaN,NaN,1,0,Royal Challengers Bangalore,field,Bengaluru,NaN,2007/08,Kolkata Knight Riders


In [2]:
# 1. Reload the data and tell Pandas to use more memory to avoid the warning
train_matches = pd.read_csv('train_IPL.csv', low_memory=False)

# 2. Create the mapping dictionary to unify rebranded franchises
team_mapping = {
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

# 3. Apply the mapping to the team columns
cols_to_clean = ['Bat First', 'Bat Second', 'toss_winner']

for col in cols_to_clean:
    if col in train_matches.columns:
        # Map the new names, but keep the old name if it's not in our dictionary (like CSK or MI)
        train_matches[col] = train_matches[col].map(team_mapping).fillna(train_matches[col])

print("Data loaded cleanly and team names standardized!")

# Let's verify it worked by checking the unique team names in 'Bat First'
print("\nUnique Teams now in the dataset:")
print(train_matches['Bat First'].unique())

Data loaded cleanly and team names standardized!

Unique Teams now in the dataset:
['Kolkata Knight Riders' 'Chennai Super Kings' 'Rajasthan Royals'
 'Mumbai Indians' 'Sunrisers Hyderabad' 'Punjab Kings'
 'Royal Challengers Bengaluru' 'Delhi Capitals' 'Kochi Tuskers Kerala'
 'Pune Warriors' 'Rising Pune Supergiant' 'Gujarat Lions'
 'Lucknow Super Giants' 'Gujarat Titans']


In [3]:
# 1. Collapse the ball-by-ball data down to one row per match
# We use drop_duplicates to keep only the first row of each Match ID
match_data = train_matches.drop_duplicates(subset=['Match ID']).copy()

print(f"Successfully collapsed data. We now have {len(match_data)} distinct matches.")

# 2. Check for our target column: 'match_won_by'
if 'match_won_by' in match_data.columns:
    print("\nSuccess: Found the 'match_won_by' column!")
    
    # 3. Standardize the target column just like we did for the others
    match_data['match_won_by'] = match_data['match_won_by'].map(team_mapping).fillna(match_data['match_won_by'])
    
    print("\nHere are the top 5 teams with the most historical wins:")
    print(match_data['match_won_by'].value_counts().head(5))
else:
    print("\nUh oh, 'match_won_by' is missing. Here are the columns we actually have:")
    print(match_data.columns.tolist())

Successfully collapsed data. We now have 1145 distinct matches.

Success: Found the 'match_won_by' column!

Here are the top 5 teams with the most historical wins:
match_won_by
Mumbai Indians                 149
Chennai Super Kings            140
Kolkata Knight Riders          134
Royal Challengers Bengaluru    128
Sunrisers Hyderabad            119
Name: count, dtype: int64


In [4]:
# 1. Create a simple column: Did the team batting first win? (1 for Yes, 0 for No)
# We check if the 'match_won_by' team is the exact same as the 'Bat First' team
match_data['bat_first_won'] = (match_data['match_won_by'] == match_data['Bat First']).astype(int)

# 2. Group the matches to calculate the historical win rate
# We group by the two teams playing, and calculate the 'mean' (average) of our 1s and 0s
h2h_stats = match_data.groupby(['Bat First', 'Bat Second'])['bat_first_won'].agg(['count', 'mean']).reset_index()

# Rename the columns so they are easy to understand
h2h_stats.rename(columns={'count': 'total_h2h_matches', 'mean': 'bat_first_win_rate'}, inplace=True)

# 3. Merge (Glue) this new feature back onto our main matches dataframe
match_data = match_data.merge(h2h_stats, on=['Bat First', 'Bat Second'], how='left')

print("Success! Feature Engineered: 'bat_first_win_rate'")

# Let's look at a specific rivalry to see if our math worked: CSK vs MI
csk_vs_mi = match_data[(match_data['Bat First'] == 'Chennai Super Kings') & (match_data['Bat Second'] == 'Mumbai Indians')]
print("\nHistorical Matchup: CSK (Batting First) vs Mumbai Indians (Chasing)")
csk_vs_mi[['Bat First', 'Bat Second', 'total_h2h_matches', 'bat_first_win_rate']].head(1)

Success! Feature Engineered: 'bat_first_win_rate'

Historical Matchup: CSK (Batting First) vs Mumbai Indians (Chasing)


,Bat First,Bat Second,total_h2h_matches,bat_first_win_rate
7,Chennai Super Kings,Mumbai Indians,18.0,0.388889


In [5]:
import pandas as pd

# 1. Load the raw data cleanly
train_matches = pd.read_csv('train_IPL.csv', low_memory=False)

# 2. Define our team mapping
team_mapping = {
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

# 3. Standardize names in the raw data
cols_to_clean = ['Bat First', 'Bat Second', 'toss_winner', 'match_won_by']
for col in cols_to_clean:
    if col in train_matches.columns:
        train_matches[col] = train_matches[col].map(team_mapping).fillna(train_matches[col])

# 4. Collapse ball-by-ball rows down to one row per match
match_data = train_matches.drop_duplicates(subset=['Match ID']).copy()

# 5. Calculate the Head-to-Head Feature
match_data['bat_first_won'] = (match_data['match_won_by'] == match_data['Bat First']).astype(int)

h2h_stats = match_data.groupby(['Bat First', 'Bat Second'])['bat_first_won'].agg(['count', 'mean']).reset_index()
h2h_stats.rename(columns={'count': 'total_h2h_matches', 'mean': 'bat_first_win_rate'}, inplace=True)

# 6. Merge the feature back into our match summary dataset
match_data = match_data.merge(h2h_stats, on=['Bat First', 'Bat Second'], how='left')

print(f"Dataset ready! Total unique matches: {len(match_data)}")
print("\nVerifying Feature (CSK vs MI bat_first_win_rate):")
print(match_data[(match_data['Bat First'] == 'Chennai Super Kings') & (match_data['Bat Second'] == 'Mumbai Indians')]['bat_first_win_rate'].head(1).values[0])

Dataset ready! Total unique matches: 1145

Verifying Feature (CSK vs MI bat_first_win_rate):
0.3888888888888889


In [6]:
# 1. Group by 'Venue' to calculate historical win rates for teams batting first
# We reuse our 'bat_first_won' column (1 if team batting first won, 0 if they lost)
venue_stats = match_data.groupby('Venue')['bat_first_won'].agg(['count', 'mean']).reset_index()

# 2. Rename columns to make them clean and clear
venue_stats.rename(columns={'count': 'total_matches_at_venue', 'mean': 'venue_bat_first_win_rate'}, inplace=True)

# 3. Merge (glue) this new venue feature back into our main match_data DataFrame
match_data = match_data.merge(venue_stats, on='Venue', how='left')

print("Success! Second Feature Engineered: 'venue_bat_first_win_rate'")

# 4. Let's look at the top 5 venues where it is easiest to DEFEND a total (highest bat-first win rate)
# We filter for stadiums that have hosted at least 10 matches so the stats are reliable
reliable_venues = match_data[match_data['total_matches_at_venue'] >= 10]
top_defending_venues = reliable_venues[['Venue', 'venue_bat_first_win_rate']].drop_duplicates().sort_values(by='venue_bat_first_win_rate', ascending=False)

print("\nTop 3 Stadiums where Batting First has the highest historical win rate:")
print(top_defending_venues.head(3).to_string(index=False))

Success! Second Feature Engineered: 'venue_bat_first_win_rate'

Top 3 Stadiums where Batting First has the highest historical win rate:
                                       Venue  venue_bat_first_win_rate
Himachal Pradesh Cricket Association Stadium                  0.615385
     Maharashtra Cricket Association Stadium                  0.549020
                      MA Chidambaram Stadium                  0.538462


In [7]:
# 1. Create our Target Vector (y)
# We map the outcome: 1 if the team batting first won, 0 if they lost
y = match_data['bat_first_won']

# 2. Convert text columns into numbers (Label Encoding)
# We convert 'toss_decision' ("field" -> 0, "bat" -> 1)
match_data['toss_decision_encoded'] = match_data['toss_decision'].map({'field': 0, 'bat': 1})

# 3. Select only our pre-match engineered features for the Feature Matrix (X)
feature_columns = [
    'bat_first_win_rate', 
    'venue_bat_first_win_rate', 
    'toss_decision_encoded'
]

X = match_data[feature_columns].copy()

# 4. Handle any missing values (NaNs)
# If a team matchup or venue is brand new, its win rate might be blank (NaN).
# We fill those blanks with 0.50 (a neutral 50% guess) so the model doesn't crash.
X = X.fillna(0.50)

print("Feature preparation complete!")
print(f"X (Features Shape): {X.shape} -> {X.shape[0]} matches, {X.shape[1]} clues per match.")
print(f"y (Target Shape): {y.shape} -> {y.shape[0]} match answers.")
print("\nPreview of your final input features (X) for the first 5 matches:")
X.head()

Feature preparation complete!
X (Features Shape): (1145, 3) -> 1145 matches, 3 clues per match.
y (Target Shape): (1145,) -> 1145 match answers.

Preview of your final input features (X) for the first 5 matches:


,bat_first_win_rate,venue_bat_first_win_rate,toss_decision_encoded
0,0.454545,0.428571,0
1,0.500000,0.442623,1
2,0.571429,0.450000,1
3,0.500000,0.447154,1
4,0.294118,0.418367,1


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss

# 1. Split the data into Training (80%) and Validation (20%) sets
# FIX: Changed 'test_split' to 'test_size'
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training with {X_train.shape[0]} matches.")
print(f"Validating with {X_val.shape[0]} matches.")

# 2. Initialize the Machine Learning Model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# 3. TRAIN the model
model.fit(X_train, y_train)
print("\nModel training complete!")

# 4. Make predictions on our Validation Set (probabilities)
val_probabilities = model.predict_proba(X_val)

# 5. Evaluate Log Loss (using the probabilities of the positive class)
val_log_loss = log_loss(y_val, val_probabilities[:, 1])

print(f"\nValidation Log Loss: {val_log_loss:.4f}")

Training with 916 matches.
Validating with 229 matches.

Model training complete!

Validation Log Loss: 1.1930


In [10]:
from sklearn.calibration import CalibratedClassifierCV

# 1. Wrap our existing Random Forest model in a Calibrator
# 'isotonic' works incredibly well for tree-based models on sports data
calibrated_model = CalibratedClassifierCV(model, method='isotonic', cv='prefit')

# 2. Fit the calibrator on our Validation Set
# It learns how overconfident the model was on the mock exam
calibrated_model.fit(X_val, y_val)

# 3. Make NEW predictions using the safe, calibrated model
calibrated_probabilities = calibrated_model.predict_proba(X_val)

# 4. Calculate our new, improved Log Loss score
calibrated_log_loss = log_loss(y_val, calibrated_probabilities[:, 1])

print(f"Old Validation Log Loss: {val_log_loss:.4f}")
print(f"New Calibrated Log Loss: {calibrated_log_loss:.4f}")

Old Validation Log Loss: 1.1930
New Calibrated Log Loss: 0.6464


C:\Users\Narasimha\anaconda3\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [13]:
test_matches = pd.read_csv('public_lb_matches.csv')
print(test_matches.columns.tolist())

['match_id', 'date', 'season', 'team_a', 'team_b', 'venue', 'city', 'toss_winner', 'toss_decision']


In [14]:
# 1. Load the unseen Kaggle test matches
test_matches = pd.read_csv('public_lb_matches.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print("Loading future matches and mapping column names...")

# 2. Fix the Kaggle Column Naming Mismatch
# We rename their columns to match the exact names our machine learning model learned
rename_dict = {
    'team_a': 'Bat First', 
    'team_b': 'Bat Second',
    'venue': 'Venue'
}
test_matches.rename(columns=rename_dict, inplace=True)

# Standardize the team names (e.g., Bangalore -> Bengaluru)
for col in ['Bat First', 'Bat Second']:
    test_matches[col] = test_matches[col].map(team_mapping).fillna(test_matches[col])

# 3. Attach our Feature Engineering
test_matches = test_matches.merge(h2h_stats, on=['Bat First', 'Bat Second'], how='left')
test_matches = test_matches.merge(venue_stats, on='Venue', how='left')

# Handle the toss decision
if 'toss_decision' in test_matches.columns:
    test_matches['toss_decision_encoded'] = test_matches['toss_decision'].map({'field': 0, 'bat': 1})
else:
    test_matches['toss_decision_encoded'] = 0.5 

# 4. Create the final Test Feature Matrix (X_test)
X_test = test_matches[feature_columns].copy()

# Fill any brand new matchups or missing clues with a neutral 50/50 probability
X_test = X_test.fillna(0.50)

# 5. GENERATE THE FINAL PREDICTIONS!
final_predictions = calibrated_model.predict_proba(X_test)

# 6. Format everything perfectly for Kaggle
submission = pd.DataFrame({
    'match_id': test_matches['match_id'],
    'team1_prob': final_predictions[:, 0], # Probability of Team A losing
    'team2_prob': final_predictions[:, 1], # Probability of Team A winning
    'tie_prob': 0.0,      # Simplification for our baseline model
    'no_result_prob': 0.0 # Simplification for our baseline model
})

# Save the file to your computer!
submission.to_csv('FINAL_SUBMISSION.csv', index=False)
print("\nBOOM! 'FINAL_SUBMISSION.csv' has been generated and saved.")
print(submission.head())

Loading future matches and mapping column names...

BOOM! 'FINAL_SUBMISSION.csv' has been generated and saved.
   match_id  team1_prob  team2_prob  tie_prob  no_result_prob
0   1473488    0.505155    0.494845       0.0             0.0
1   1473489    0.505155    0.494845       0.0             0.0
2   1473490    0.505155    0.494845       0.0             0.0
3   1473491    0.616438    0.383562       0.0             0.0
4   1473492    0.616438    0.383562       0.0             0.0


In [17]:
# 1. Load the 48 known future matches and the 53-row Kaggle template
test_matches = pd.read_csv('public_lb_matches.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print("Loading future matches and mapping column names...")

# 2. Fix the Kaggle Column Naming Mismatch
rename_dict = {
    'team_a': 'Bat First', 
    'team_b': 'Bat Second',
    'venue': 'Venue'
}
test_matches.rename(columns=rename_dict, inplace=True)

# Standardize the team names
for col in ['Bat First', 'Bat Second']:
    test_matches[col] = test_matches[col].map(team_mapping).fillna(test_matches[col])

# 3. Attach our Feature Engineering
test_matches = test_matches.merge(h2h_stats, on=['Bat First', 'Bat Second'], how='left')
test_matches = test_matches.merge(venue_stats, on='Venue', how='left')

# Handle the toss decision
if 'toss_decision' in test_matches.columns:
    test_matches['toss_decision_encoded'] = test_matches['toss_decision'].map({'field': 0, 'bat': 1})
else:
    test_matches['toss_decision_encoded'] = 0.5 

# 4. Create the final Test Feature Matrix (X_test)
X_test = test_matches[feature_columns].copy().fillna(0.50)

# 5. GENERATE PREDICTIONS FOR THE 48 MATCHES
final_predictions = calibrated_model.predict_proba(X_test)

# Create a temporary dataframe with just our 48 predictions
predictions_df = pd.DataFrame({
    'match_id': test_matches['match_id'],
    'team1_prob': final_predictions[:, 0], 
    'team2_prob': final_predictions[:, 1]
})

# 6. FIX DATA TYPE MISMATCH (Force both to string so they merge flawlessly)
sample_sub['match_id'] = sample_sub['match_id'].astype(str)
predictions_df['match_id'] = predictions_df['match_id'].astype(str)

# 7. THE BULLETPROOF KAGGLE TRICK
# Merge our predictions into the official 53-row template
final_sub = sample_sub[['match_id']].merge(predictions_df, on='match_id', how='left')

# The 5 missing playoff matches will now be empty (NaN). Fill them with safe 0.50 predictions!
final_sub['team1_prob'] = final_sub['team1_prob'].fillna(0.50)
final_sub['team2_prob'] = final_sub['team2_prob'].fillna(0.50)
final_sub['tie_prob'] = 0.0
final_sub['no_result_prob'] = 0.0

# Save the file!
final_sub.to_csv('FINAL_SUBMISSION_53.csv', index=False)
print(f"\nBOOM! 'FINAL_SUBMISSION_53.csv' has been generated with exactly {len(final_sub)} rows!")
print(final_sub.head())

Loading future matches and mapping column names...

BOOM! 'FINAL_SUBMISSION_53.csv' has been generated with exactly 53 rows!
  match_id  team1_prob  team2_prob  tie_prob  no_result_prob
0  1473488    0.505155    0.494845       0.0             0.0
1  1473489    0.505155    0.494845       0.0             0.0
2  1473490    0.505155    0.494845       0.0             0.0
3  1473491    0.616438    0.383562       0.0             0.0
4  1473492    0.616438    0.383562       0.0             0.0
